## Table of Contents

1.  [Introduction](#Introduction)
2.  [Data Loading](#Data-Loading)
3.  [Initial Data Exploration](#Initial-Data-Exploration)
    - [Viewing Data Head](#Viewing-Data-Head)
    - [Checking Data Types and Nulls](#Checking-Data-Types-and-Nulls)
4.  [Data Aggregation](#Data-Aggregation)
    - [Grouping by Assignment Group](#Grouping-by-Assignment-Group)
5.  [Data Visualization](#Data-Visualization)
    - [Bar Chart of Ticket Count by Assignment Group](#Bar-Chart-of-Ticket-Count-by-Assignment-Group)
6.  [Conclusion](#Conclusion)

## Overview
This notebook does the data preparation for the analytics exercise. Please see the [example narrative](example_narrative.md) for details of the analytics task and process used to develop the solution. The first step in the process is exploratory data analysis. This phase consists of the following steps:
1. Read the raw data set
2. Select the subset of attributes that are need for this use case
3. Analyze the (subsetted) dataset for data quality
4. Fix attribute noise
5. Write the denoised data for the next phase in the workflow.
6. Log the relevance and noise processing details done as part of exploratory data analysis to KMDS


## Read Data

In [35]:
from importlib.resources import files
import pandas as pd
fp = str(files("kmds.examples").joinpath("incident_event_log_02.csv"))
df = pd.read_csv(fp)

/tmp/ipykernel_12293/2247844116.py:4: DtypeWarning: Columns (0: cmdb_ci, 1: caused_by) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(fp)


## Raw Data Report from Data Profiler

In [36]:
# Create a raw data profile report
from dataprofiler import Data, Profiler
from pathlib import Path

# 1. Load the data
data = Data(fp)

# 2. Run the Profiler
profile = Profiler(data)

# 3. Generate the report (use "pretty" format for a readable dictionary)
report = profile.report(report_options={"output_format": "pretty"})

# 4. Flatten and save global stats
g_stats = report.get("global_stats", {})
global_df = pd.DataFrame(list(g_stats.items()), columns=["Global Metric", "Value"])
global_md = global_df.to_markdown(index=False)

# 5. Flatten and save column stats
all_columns_data = []
for col in report.get("data_stats", []):
    base_info = {k: v for k, v in col.items() if k != "statistics"}
    stats_info = col.get("statistics", {})
    combined_row = {**base_info, **stats_info}
    all_columns_data.append(combined_row)

detailed_col_df = pd.DataFrame(all_columns_data).set_index("column_name").T
detailed_col_md = detailed_col_df.to_markdown()

# 6. Final markdown assembly
full_markdown = f"""
# Complete Data Profile Report

## Global Dataset Statistics
{global_md}

## Detailed Column-by-Column Statistics
{detailed_col_md}
"""

Path("reports").mkdir(parents=True, exist_ok=True)
fpr = "reports/full_data_profile.md"
with open(fpr, "w", encoding="utf-8") as f:
    f.write(full_markdown)

print(f"Full report saved to {fpr}")


/home/rajiv/programming/kmds_maintainence/KMDS/.venv/lib/python3.13/site-packages/dataprofiler/profilers/profile_builder.py:757: RuntimeWarning: 

!!! WARNING Partial Profiler Failure !!!

Profiling Type: data_labeler
Exception: ModuleNotFoundError
Message: No module named 'tensorflow'

For labeler errors, try installing the extra ml requirements via:

$ pip install dataprofiler[ml] --user


  profiler_utils.warn_on_profile("data_labeler", e)


INFO:DataProfiler.profilers.profile_builder: Finding the Null values in the columns...  (with 15 processes)


/home/rajiv/programming/kmds_maintainence/KMDS/.venv/lib/python3.13/site-packages/dataprofiler/profilers/profile_builder.py:2903: UserWarning: The data will be profiled with a sample size of 28342. All statistics will be based on this subsample and not the whole dataset.
  warnings.warn(
100%|██████████| 36/36 [00:00<00:00, 40.18it/s]


INFO:DataProfiler.profilers.profile_builder: Calculating the statistics...  (with 4 processes)


100%|██████████| 36/36 [00:05<00:00,  6.23it/s]


Full report saved to reports/full_data_profile.md


## Select Columns Needed

In [37]:
df.head()

,number,incident_state,active,reassignment_count,reopen_count,sys_mod_count,made_sla,caller_id,opened_by,opened_at,...,u_priority_confirmation,notify,problem_id,rfc,vendor,caused_by,closed_code,resolved_by,resolved_at,closed_at
0,INC0000045,New,True,0,0,0,True,Caller 2403,Opened by 8,29/2/2016 01:16,...,False,Do Not Notify,NaN,NaN,NaN,NaN,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
1,INC0000045,Resolved,True,0,0,2,True,Caller 2403,Opened by 8,29/2/2016 01:16,...,False,Do Not Notify,NaN,NaN,NaN,NaN,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
2,INC0000045,Resolved,True,0,0,3,True,Caller 2403,Opened by 8,29/2/2016 01:16,...,False,Do Not Notify,NaN,NaN,NaN,NaN,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
3,INC0000045,Closed,False,0,0,4,True,Caller 2403,Opened by 8,29/2/2016 01:16,...,False,Do Not Notify,NaN,NaN,NaN,NaN,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
4,INC0000047,New,True,0,0,0,True,Caller 2403,Opened by 397,29/2/2016 04:40,...,False,Do Not Notify,NaN,NaN,NaN,NaN,code 5,Resolved by 81,1/3/2016 09:52,6/3/2016 10:00


In [38]:
SELECT_COLS = ['number', 'sys_created_at', 'closed_at', 'assignment_group']
closed_tickets = df.incident_state == "Closed"
df_closed_tickets = df[closed_tickets][SELECT_COLS].copy()
del df
df_closed_tickets = df_closed_tickets.reset_index(drop=True)

In [39]:
import dateutil
from dateutil.parser import parse


## Inspect Null Information in Dataset

In [40]:
df_closed_tickets.info()

<class 'pandas.DataFrame'>
RangeIndex: 24985 entries, 0 to 24984
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   number            24985 non-null  str  
 1   sys_created_at    13466 non-null  str  
 2   closed_at         24985 non-null  str  
 3   assignment_group  22816 non-null  str  
dtypes: str(4)
memory usage: 1.7 MB


## Noise Observations

1. This use case computes the time to resolve a ticket. This is time that has elapsed between the ticket creation and the ticket closed time stamps. Only closed tickets and tickets with valid values for both these timestamps are used.
2. There is inconsistency in the datetime format for both ticket creation and ticket closed times. The parse function from the dateutil library is used to parse times into ISO format and then use that form for the calculation of the time to resolution
   

## Noise Filter #1 definition

In [41]:
def noise_filter_1(row):
    creation_date = row['sys_created_at']
    valid_creation_date = True
    valid_closed_date = True
    cd = cls_dt = None
    if pd.isna(creation_date):
        valid_creation_date = False
    else:
        try:
            creation_date = parse(creation_date)
        except:
            valid_creation_date = False
    
            
    closed_date = row["closed_at"]
    
    if pd.isna(closed_date):
        valid_closed_date = False
    else:
        try:
            cls_dt = parse(closed_date)

        except:
            valid_closed_date = False
    
    
    clean_row = valid_creation_date & valid_closed_date
    

    return clean_row


## Apply Noise Filter #1

In [42]:
df_closed_tickets = df_closed_tickets[df_closed_tickets.apply(noise_filter_1, axis=1)]

In [43]:
df_closed_tickets["sys_created_at"] = df_closed_tickets["sys_created_at"].apply(parse) 

In [44]:
df_closed_tickets["closed_at"] = df_closed_tickets["closed_at"].apply(parse) 

## Define Noise Filter #2

In [45]:
dict_types = {"number": 'str', "sys_created_at": 'datetime64[ns]',
              "closed_at": "datetime64[ns]" , "assignment_group": 'str'}
df_closed_tickets = df_closed_tickets.astype(dict_types)
valid_closing_dates = df_closed_tickets["closed_at"] > df_closed_tickets["sys_created_at"]
q2_2016 = (df_closed_tickets.closed_at.dt.quarter == 2) & \
        (df_closed_tickets.closed_at.dt.year == 2016)
in_range_good_tickets = q2_2016 & valid_closing_dates

## Apply Noise Filter #2

In [46]:

df_closed_tickets = df_closed_tickets[in_range_good_tickets].reset_index(drop=True)

In [47]:
df_closed_tickets

,number,sys_created_at,closed_at,assignment_group
0,INC0000045,2016-02-29 01:23:00,2016-05-03 12:00:00,Group 56
1,INC0000047,2016-02-29 04:57:00,2016-06-03 10:00:00,Group 24
2,INC0000062,2016-02-29 07:26:00,2016-05-03 16:00:00,Group 23
3,INC0000063,2016-02-29 07:17:00,2016-05-03 17:00:00,Group 23
4,INC0000071,2016-02-29 08:17:00,2016-06-03 11:00:00,Group 24
...,...,...,...,...
3802,INC0041652,2016-06-24 08:43:00,2016-06-29 15:00:00,Group 55
3803,INC0082574,2016-01-11 10:57:00,2016-06-11 12:07:00,Group 3
3804,INC0082685,2016-01-11 14:37:00,2016-05-12 16:00:00,Group 31
3805,INC0091855,2016-01-12 10:46:00,2016-06-12 11:00:00,Group 22


## Relevance Observations
1. Only (number', 'sys_created_at', 'closed_at', 'assignment_group') are needed for this analysis
2. Only ticket activity in the second quarter of 2016 is needed for this analysis

## Write Report Data

In [48]:
fp = str(files("kmds.examples").joinpath("q2_2016_ticket_resolution_data.csv"))
df_closed_tickets.to_csv(fp, index=False)

## Data Profiler Summary for Cleaned Data

In [49]:

# 1. Load the data
data = Data(fp)

# 2. Run the Profiler
profile = Profiler(data)

# 3. Generate the report (use "pretty" format for a readable dictionary)
report = profile.report(report_options={"output_format": "pretty"})

# 2. Flatten and Save Global Stats
g_stats = report.get('global_stats', {})
global_df = pd.DataFrame(list(g_stats.items()), columns=["Global Metric", "Value"])
global_md = global_df.to_markdown(index=False)

# 3. Flatten and Save Column Stats (Capturing Nested Statistics)
all_columns_data = []
for col in report.get('data_stats', []):
    base_info = {k: v for k, v in col.items() if k != 'statistics'}
    stats_info = col.get('statistics', {})
    combined_row = {**base_info, **stats_info}
    all_columns_data.append(combined_row)

detailed_col_df = pd.DataFrame(all_columns_data).set_index('column_name').T
detailed_col_md = detailed_col_df.to_markdown()

# 4. Final Markdown Assembly
full_markdown = f"""
# Complete Data Profile Report

## Global Dataset Statistics
{global_md}

## Detailed Column-by-Column Statistics
{detailed_col_md}
"""
fpr = "reports/analytics_data_profile.md"
with open(fpr, "w", encoding="utf-8") as f:
    f.write(full_markdown)

print(f"Full report saved to {fpr}")


INFO:DataProfiler.profilers.profile_builder: Finding the Null values in the columns... 


/home/rajiv/programming/kmds_maintainence/KMDS/.venv/lib/python3.13/site-packages/dataprofiler/profilers/profile_builder.py:757: RuntimeWarning: 

!!! WARNING Partial Profiler Failure !!!

Profiling Type: data_labeler
Exception: ModuleNotFoundError
Message: No module named 'tensorflow'

For labeler errors, try installing the extra ml requirements via:

$ pip install dataprofiler[ml] --user


  profiler_utils.warn_on_profile("data_labeler", e)
100%|██████████| 4/4 [00:00<00:00, 155.13it/s]

INFO:DataProfiler.profilers.profile_builder: Calculating the statistics... 



100%|██████████| 4/4 [00:00<00:00, 26.46it/s]


Full report saved to reports/analytics_data_profile.md


In [50]:
report

{'global_stats': {'samples_used': 3807,
  'column_count': 4,
  'row_count': 3807,
  'row_has_null_ratio': 0.1122,
  'row_is_null_ratio': 0.0,
  'unique_row_ratio': 0.9974,
  'duplicate_row_count': 10,
  'file_type': 'csv',
  'encoding': 'utf-8',
  'correlation_matrix': None,
  'chi2_matrix': '[[nan, nan, nan, nan],\n [nan, nan, nan, nan],\n [nan, nan,  1.,  0.],\n [nan, nan,  0.,  1.]]',
  'profile_schema': {'number': [0],
   'sys_created_at': [1],
   'closed_at': [2],
   'assignment_group': [3]},
  'times': {'row_stats': 0.0047}},
 'data_stats': [{'column_name': 'number',
   'data_type': 'string',
   'categorical': False,
   'order': 'ascending',
   'samples': "['INC0019097', 'INC0020907', 'INC0002509', 'INC0000702', 'INC0030121']",
   'statistics': {'min': 10.0,
    'max': 10.0,
    'mode': '[10.]',
    'median': 10.0,
    'sum': 38070.0,
    'mean': 10.0,
    'variance': 0.0,
    'stddev': 0.0,
    'skewness': 0.0,
    'kurtosis': -3.0024,
    'histogram': {'bin_counts': '[3807]', '

In [51]:
df_closed_tickets.columns

Index(['number', 'sys_created_at', 'closed_at', 'assignment_group'], dtype='str')

In [52]:
dtypes_meta = {"attribute": [], "type": []}
for k, v in dict_types.items():
    dtypes_meta["attribute"].append(k)
    dtypes_meta["type"].append(v)
    
df_dtypes = pd.DataFrame.from_dict(dtypes_meta)
fp_types = str(files("kmds.examples").joinpath("ticket_resolution_dtypes.csv"))
df_dtypes.to_csv(fp_types, index=False)

## Log Exploratory Data Analysis Observations to KMDS Knowledge Base

In [53]:
from kmds.ontology.kmds_ontology import *
from kmds.tagging.tag_types import ExploratoryTags

## Note:
1. In this example the base ontology that comes with the application (onto) is used as the namespace to log all observations into.
2. This ontology is saved to a directory on the repo. This can be a URL or another location - adding this functionality is quite straight forward, but for illustration, a directory is used.
3. The saved ontology is used in all subsequent sessions by **loading** this ontology and using that as the namespace to log all observations.

   If you choose to work through these examples, please take a minute to review how ontologies are saved and loaded.

In [54]:
kaw = KnowledgeApplicationWorkflow("ITSM modelling", namespace=onto)

In [55]:
exp_obs_list = []
observation_count :int = 1
e1 = ExploratoryObservation(namespace=onto)


In [56]:
from kmds.ontology.intent_types import IntentType
from kmds.utils.natural_language_observation import map_text_to_observation

In [57]:
e1.finding = "Only (number', 'sys_created_at', 'closed_at', 'assignment_group') are needed for this analysis"
e1.finding_sequence = observation_count
e1.exploratory_observation_type = ExploratoryTags.RELEVANCE_OBSERVATION.value
e1.intent = IntentType.DATA_UNDERSTANDING.value
exp_obs_list.append(e1)

In [58]:
observation_count += 1
e2 = ExploratoryObservation(namespace=onto)
e2.finding = "Only ticket activity in the second quarter of 2016 is needed for this analysis"
e2.finding_sequence = observation_count
e2.exploratory_observation_type = ExploratoryTags.RELEVANCE_OBSERVATION.value
e2.intent = IntentType.DATA_UNDERSTANDING.value
exp_obs_list.append(e2)

In [59]:
observation_count += 1
e3 = ExploratoryObservation(namespace=onto)
e3.finding = "This use case computes the time to resolve a ticket. This is time that has elapsed between the ticket creation and the ticket\
closed time stamps. Only closed tickets andtickets with valid values for both these timestamps are used."
e3.finding_sequence = observation_count
e3.exploratory_observation_type = ExploratoryTags.DATA_QUALITY_OBSERVATION.value
e3.intent = IntentType.DATA_UNDERSTANDING.value
exp_obs_list.append(e3)

In [60]:
observation_count += 1
nl_mapping = map_text_to_observation(
    "Ticket creation and closed timestamps have inconsistent datetime formats, so they must be normalized before calculating time to resolution."
)
e4 = ExploratoryObservation(namespace=onto)
e4.finding = nl_mapping.finding
e4.finding_sequence = observation_count
e4.exploratory_observation_type = nl_mapping.observation_type
e4.intent = nl_mapping.intent
exp_obs_list.append(e4)

In [61]:
kaw.has_exploratory_observations = exp_obs_list

In [62]:
type(e4.intent)

str

In [63]:
from owlready2 import *
from kmds.utils.path_utils import get_package_kb_path
from importlib.resources import files
KNOWLEDGE_BASE = str(files("kmds.examples").joinpath("example_analytics_kb_app_workflow.xml"))
onto.save(file=KNOWLEDGE_BASE, format="rdfxml")

## Plot/Image Ingest Parity (Notebook)

This section adds notebook analogues for document ingest:
1. ingest a plot image,
2. generate an observation draft,
3. retag the draft,
4. log the final observation to the KMDS KB.

In [64]:
from pathlib import Path
from kmds.dashboard.service import summarize_plot_image, preview_analyst_observation, log_analyst_observation

def ingest_plot_image(image_path: str, goal: str, model: str = "llama3.2-vision") -> dict:
    """Ingest an image and return a plot summary draft + mapping preview."""
    path = Path(image_path).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {path}")

    result = summarize_plot_image(
        image_name=path.name,
        image_bytes=path.read_bytes(),
        goal=goal,
        model=model,
    )
    return result

def draft_plot_observation(observation_text: str, phase: str | None = None) -> dict:
    """Preview ontology mapping for an editable plot-derived observation."""
    return preview_analyst_observation(observation_text, phase=phase)

def retag_and_log_plot_observation(
    observation_text: str,
    project_file: str,
    workflow_name: str,
    phase: str | None = "Exploratory",
    workflow_type: str | None = "application",
    create_project: bool = False,
    ) -> dict:
    """Persist the observation into KMDS after edit/retag."""
    return log_analyst_observation(
        text=observation_text,
        phase=phase,
        workflow_name=workflow_name,
        project_file=project_file,
        workflow_type=workflow_type,
        create_project=create_project,
    )

### Usage Example

Set `PLOT_IMAGE_PATH` to a real chart image and run the cell below.

Requirements:
- Ollama running locally (`http://127.0.0.1:11434`)
- vision model available, e.g. `ollama pull llama3.2-vision` or `ollama pull chartllama`.

In [65]:
PLOT_IMAGE_PATH = ""  # e.g., "/absolute/path/to/eda_plot.png"
PLOT_GOAL = "Summarize EDA chart patterns, anomalies, and operational implications."
PLOT_MODEL = "llama3.2-vision"  # or "chartllama"

if PLOT_IMAGE_PATH:
    plot_ingest = ingest_plot_image(
        image_path=PLOT_IMAGE_PATH,
        goal=PLOT_GOAL,
        model=PLOT_MODEL,
    )
    
    print("Generated draft:")
    print(plot_ingest["observation_text"])
    
    # Edit if needed before retag/logging.
    edited_observation = plot_ingest["observation_text"]
    
    # Retag preview (same pattern as document-ingest draft workflow).
    mapping_preview = draft_plot_observation(
        observation_text=edited_observation,
        phase="Exploratory",
    )
    print("\nMapping preview:")
    print(mapping_preview)
    
    # Persist to KB (uses current notebook KB variable when available).
    project_fp = KNOWLEDGE_BASE if "KNOWLEDGE_BASE" in globals() else "example_analytics_kb_app_workflow.xml"
    log_result = retag_and_log_plot_observation(
        observation_text=edited_observation,
        project_file=project_fp,
        workflow_name="ITSM modelling",
        phase="Exploratory",
        workflow_type="application",
        create_project=False,
    )
    print("\nLogged:")
    print(log_result)
else:
    print("Set PLOT_IMAGE_PATH to run the plot-ingest analogue flow.")

Set PLOT_IMAGE_PATH to run the plot-ingest analogue flow.
